In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

In [ ]:
def load_data(file):
    with fits.open(file) as hdul:
        pheader = hdul[0].header
        if str(file).endswith("raw.fits"):
            data = hdul[1].data
        if str(file).endswith("bias.fits"):
            data = hdul[2].data
        elif '2507' in str(file):
            data = hdul[0].data
        else:
            data = hdul[1].data if len(hdul) > 1 else hdul[0].data   
        dheader = hdul[1].header if len(hdul) > 1 else hdul[0].header
    return data, pheader, dheader

In [ ]:
def load_file(filelist):
    file_dict = {}
    
    for file in filelist:
        data, pheader, dheader = load_data(file)
        file_dict[str(file)] = (data, pheader, dheader)
    return file_dict

In [ ]:
def blev_parsing(data):
    if data.ndim != 2:
        if data.ndim == 3:
            data = data[:, :, 1]
    area = data[9:790, 8:14]

    oddv = area[:, 0::2]
    evenv = area[:, 1::2]
    return np.mean(oddv), np.mean(evenv)

In [ ]:
def apply_bias(raw, blev):
    mean_o, mean_e = blev_parsing(blev)
    corr_data = raw.astype('float64')
    corr_data[:, 0::2] -= mean_o
    corr_data[:, 1::2] -= mean_e
    return corr_data

In [ ]:
def calibration(dpath):
    files = load_file(dpath)
    for filename, file_tuple in files.items():
        if 'raw' in filename:
            raw_data = file_tuple[0]
            exptime = file_tuple[1]['EXPTIME']
            gain = file_tuple[1]['ATODGAIN']
        elif 'bias' in filename:
            bias_data = file_tuple[0]
        elif 'mask' in filename:
            mask_data = file_tuple[0]
        elif 'blev' in filename:
            blev_data = file_tuple[0]
        elif 'dark' in filename:
            dark_data = file_tuple[0]
        elif 'flat' in filename:
            flat_data = file_tuple[0]

    corr_blev = apply_bias(raw_data, blev_data)
    #corr_bias = corr_bev - bias_data
    #dark_scaled = dark_data * exptime
    #corr_dark = corr_bias - dark_scaled
    #corr_flat = corr_dark * flat_data
    return bias_data

In [ ]:
here = pathlib.Path('.')
ubai2505m = list(here.rglob('**/ubai2505m/**/*.fits'))
ubai2507m = list(here.rglob('**/ubai2507m/**/*.fits'))
ubai2508m = list(here.rglob('**/ubai2508m/**/*.fits'))
print(ubai2505m )

In [ ]:
calibration(ubai2508m)